# NBA Player Longevity Prediction
## Naive Bayes Classification — Gaussian Naive Bayes Model

**Project Overview:**  
In this project, we analyse engineered NBA player data using Python and Scikit-learn to build a **Gaussian Naive Bayes classification model**. The goal is to predict whether a player will remain in the NBA for 5 or more years (`target_5yrs`), evaluate performance using Confusion Matrix, Precision, and Recall, and translate statistical predictions into actionable scouting insights.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, accuracy_score,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

## 2. Load Dataset & Confirm Target Variable

In [ ]:
df = pd.read_csv('extracted_nba_players_data.csv')
print('Dataset Shape:', df.shape)
df.head()

In [ ]:
print('Column Names & Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())
print('\nTotal missing values:', df.isnull().sum().sum())

In [ ]:
# Confirm target variable
print('Target Variable — target_5yrs:')
print(df['target_5yrs'].value_counts())
print()
print(df['target_5yrs'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

In [ ]:
# Visualise target distribution
fig, ax = plt.subplots(figsize=(6, 4))
df['target_5yrs'].value_counts().plot(
    kind='bar', ax=ax,
    color=['#e74c3c', '#2ecc71'],
    edgecolor='black'
)
ax.set_title('Target Variable Distribution (target_5yrs)', fontsize=13)
ax.set_xlabel('Career ≥5 Years (1 = Yes, 0 = No)')
ax.set_ylabel('Number of Players')
ax.set_xticklabels(['Did NOT last 5 yrs (0)', 'Lasted 5+ yrs (1)'], rotation=0)
plt.tight_layout()
plt.show()

**Target Variable Confirmation:**

`target_5yrs` is the **dependent variable** (binary classification target):
- `1` = Player remained in the NBA for **5 or more years** (831 players, 62.0%)
- `0` = Player did **not** reach 5 years (509 players, 38.0%)

This dataset is the **engineered output** from the feature engineering pipeline — it contains only continuous numeric metrics compatible with Gaussian Naive Bayes, which assumes each feature follows a Gaussian (normal) distribution.

## 3. Exploratory Data Analysis

In [ ]:
print('Descriptive Statistics:')
df.describe().round(3)

In [ ]:
# Average stats by target class
feature_cols = [c for c in df.columns if c != 'target_5yrs']
avg_by_target = df.groupby('target_5yrs')[feature_cols].mean().T.round(3)
avg_by_target.columns = ['Did Not Last 5 Yrs (0)', 'Lasted 5+ Yrs (1)']
print('Average Feature Values by Career Longevity:')
avg_by_target

In [ ]:
# Feature distributions by target class
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for ax, col in zip(axes.flatten(), feature_cols):
    for val, color, label in [(0, '#e74c3c', 'Did Not Last 5 Yrs'), (1, '#2ecc71', 'Lasted 5+ Yrs')]:
        df[df['target_5yrs'] == val][col].hist(
            ax=ax, bins=25, alpha=0.6, color=color, label=label, edgecolor='white'
        )
    ax.set_title(col, fontsize=10)
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)

plt.suptitle('Feature Distributions by Career Longevity (Gaussian Assumption Check)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of features
plt.figure(figsize=(10, 7))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    annot_kws={'size': 9}
)
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Preprocess Features for Gaussian Naive Bayes

In [ ]:
# Define feature matrix X and target vector y
feature_cols = [c for c in df.columns if c != 'target_5yrs']

X = df[feature_cols]
y = df['target_5yrs']

print('Features used for modelling:')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')
print(f'\nFeature matrix shape : {X.shape}')
print(f'Target vector shape  : {y.shape}')
print(f'All features numeric : {X.dtypes.apply(lambda d: d in ["float64", "int64"]).all()}')

**Preprocessing Notes for Gaussian Naive Bayes:**

Gaussian Naive Bayes assumes that each feature follows a **continuous Gaussian (normal) distribution** within each class. This dataset is already well-suited because:
- All features are **continuous numeric metrics** (no categorical columns)
- The dataset has **zero missing values**
- Features like `fg`, `ft`, `3p` are percentages and `efficiency`, `total_points` are derived continuous metrics

**Note:** Unlike Logistic Regression, Gaussian Naive Bayes does **not** require feature scaling (StandardScaler), because it models the distribution of each feature independently using mean and variance — scaling does not change the underlying Gaussian shape.

## 5. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(y)*100:.0f}%)')
print(f'Test set     : {X_test.shape[0]:,} rows ({X_test.shape[0]/len(y)*100:.0f}%)')
print(f'\nClass distribution in training set:')
print(y_train.value_counts())
print(f'\nClass distribution in test set:')
print(y_test.value_counts())

## 6. Build & Train Gaussian Naive Bayes Model

In [ ]:
# Instantiate and train the Gaussian Naive Bayes classifier
gnb = GaussianNB()
gnb.fit(X_train, y_train)

print('Gaussian Naive Bayes model trained successfully.')
print('Classes:', gnb.classes_)
print('Class priors (from training data):', gnb.class_prior_.round(4))

In [ ]:
# View per-class Gaussian parameters (mean per feature)
theta_df = pd.DataFrame(
    gnb.theta_,
    columns=feature_cols,
    index=['Class 0 (Did Not Last)', 'Class 1 (Lasted 5+ Yrs)']
).round(4)

print('Class-Conditional Feature Means (θ):')
theta_df

## 7. Model Evaluation

In [ ]:
# Generate predictions
y_pred = gnb.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)

print('='*45)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}  ({prec*100:.2f}%)')
print(f'  Recall    : {rec:.4f}  ({rec*100:.2f}%)')
print('='*45)

In [ ]:
print('Full Classification Report:')
print(classification_report(
    y_test, y_pred,
    target_names=['Did Not Last 5 Yrs (0)', 'Lasted 5+ Yrs (1)']
))

In [ ]:
# Confusion Matrix — values and visualisation
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('Confusion Matrix:')
print(cm)
print()
print(f'True Negatives  (correctly predicted: did NOT last) : {tn}')
print(f'False Positives (predicted lasted, actually did not): {fp}  ← "Busts" (costly for scouts)')
print(f'False Negatives (predicted did not last, actually did): {fn}  ← Missed talent')
print(f'True Positives  (correctly predicted: lasted 5+ yrs): {tp}')

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Did Not Last 5 Yrs', 'Lasted 5+ Yrs']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Gaussian Naive Bayes', fontsize=13)
plt.tight_layout()
plt.show()

**Model Performance Summary:**

| Metric | Score |
|--------|-------|
| Accuracy | 66.42% |
| Precision | 86.92% |
| Recall | 55.03% |

**Business Interpretation (aligned with scouting priorities):**

- **Precision (86.92%):** When the model predicts a player *will* last 5+ years, it is correct ~87% of the time. This is highly valuable for scouts — it keeps the **"bust" rate low** (only 14 false positives in this test set). Teams spend significant resources on drafted players, so high precision minimises wasted investment.

- **Recall (55.03%):** The model only catches ~55% of players who *actually* lasted 5+ years, missing 76 talented players. This is the key **trade-off**: the model is conservative — it correctly avoids busts, but at the cost of missing genuine long-term talent.

- **For scouting departments:** If the priority is **avoiding busts** (wasted contracts), trust the model's positive predictions — they are 87% reliable. If the priority is **not missing talent**, supplement the model with human scouting for players it flags as negative.

## 8. Most Influential Features Driving Longevity Predictions

In [ ]:
# Feature importance: correlation with target + mean difference between classes
corr_with_target = df[feature_cols].corrwith(df['target_5yrs']).abs().sort_values(ascending=False)

print('Feature Correlation with target_5yrs (absolute):')
print(corr_with_target.round(4).to_string())

In [ ]:
# Mean difference between classes — how much each feature separates the two groups
class_means = df.groupby('target_5yrs')[feature_cols].mean()
mean_diff = (class_means.loc[1] - class_means.loc[0]).abs().sort_values(ascending=False)

print('Mean Difference Between Classes (Class 1 − Class 0, absolute):')
print(mean_diff.round(4).to_string())

In [ ]:
# Visualise both importance metrics side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation chart
corr_with_target.plot(kind='barh', ax=axes[0], color='#3498db', edgecolor='black')
axes[0].set_title('Feature Correlation with target_5yrs', fontsize=11)
axes[0].set_xlabel('Absolute Pearson Correlation')
axes[0].invert_yaxis()

# Mean difference chart
mean_diff.plot(kind='barh', ax=axes[1], color='#e67e22', edgecolor='black')
axes[1].set_title('Mean Difference Between Classes', fontsize=11)
axes[1].set_xlabel('|Mean(Class 1) − Mean(Class 0)|')
axes[1].invert_yaxis()

plt.suptitle('Most Influential Features Driving Longevity Predictions', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Naive Bayes Independence Assumption — Analysis in Sports Context

In [ ]:
# Demonstrate the correlation violations of the independence assumption
corr_features = df[feature_cols].corr().abs()

# Find strongly correlated feature pairs
upper = corr_features.where(np.triu(np.ones(corr_features.shape), k=1).astype(bool))
corr_pairs = [
    (col, row, round(upper.loc[row, col], 3))
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.5
]
corr_pairs.sort(key=lambda x: -x[2])

print('Feature pairs with correlation > 0.50 (independence assumption violations):')
print(f'{"Feature A":<18} {"Feature B":<18} {"Correlation":>12}')
print('-' * 50)
for a, b, c in corr_pairs:
    print(f'{a:<18} {b:<18} {c:>12}')

**The Naive Bayes Independence Assumption — Critical Assessment:**

Gaussian Naive Bayes assumes that **all features are conditionally independent** of each other given the class label. In plain terms: knowing a player's `total_points` should give you no additional information about their `reb` or `tov`, once you know whether they lasted 5 years.

**Is this realistic for basketball statistics? No — and here's why:**

Basketball stats are structurally correlated:
- Players who score more (`total_points`) naturally have more rebounds and turnovers — they handle the ball more
- `efficiency` was engineered *from* `total_points`, `reb`, `ast`, `stl`, `blk`, and `tov` — these features share information by construction
- `tov` and `ast` are both ball-handling metrics and tend to move together

**Impact on model validity:**
- When the independence assumption is violated, Naive Bayes becomes **overconfident** in its probability estimates — it counts correlated information multiple times
- However, despite this, Gaussian Naive Bayes often still performs well in **classification** (not probability estimation) tasks, even with correlated features
- The **87% Precision** suggests the model has learned to separate classes effectively, even if its probability outputs are not perfectly calibrated

**When to trust the model:**
- Trust it for **positive predictions** (87% Precision) — when it says a player will last, it is usually right
- Question it for **negative predictions** (only 55% Recall) — the model misses nearly half of true long-term players
- Do **not** rely on the raw probability scores as calibrated probabilities; use them for ranking only

## 10. Stakeholder Summary & Scouting Recommendations

In [ ]:
# Predict probabilities on test set for scouting use
y_proba = gnb.predict_proba(X_test)

# Build a results table
results_df = pd.DataFrame({
    'Actual'           : y_test.values,
    'Predicted'        : y_pred,
    'Prob_Not_Last_5yr': y_proba[:, 0].round(3),
    'Prob_Last_5yr'    : y_proba[:, 1].round(3)
})

print('Sample Prediction Results (first 15 rows):')
print(results_df.head(15).to_string(index=False))

In [ ]:
# Probability distribution of predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for val, color, label in [(0, '#e74c3c', 'Actual: Did Not Last'), (1, '#2ecc71', 'Actual: Lasted 5+ Yrs')]:
    subset = results_df[results_df['Actual'] == val]
    subset['Prob_Last_5yr'].hist(
        ax=axes[0], bins=20, alpha=0.6, color=color, label=label, edgecolor='white'
    )

axes[0].set_title('Predicted Probability Distribution\n(by Actual Class)', fontsize=11)
axes[0].set_xlabel('Predicted Probability of Lasting 5+ Years')
axes[0].set_ylabel('Count')
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Decision threshold (0.5)')
axes[0].legend()

# High-confidence correct predictions
high_conf_correct = results_df[
    (results_df['Actual'] == results_df['Predicted']) &
    (results_df['Prob_Last_5yr'] > 0.75)
]
low_conf = results_df[
    (results_df['Prob_Last_5yr'] >= 0.4) &
    (results_df['Prob_Last_5yr'] <= 0.6)
]

categories = ['High-confidence\ncorrect (>75%)', 'Borderline\n(40–60%)', 'All predictions']
counts = [len(high_conf_correct), len(low_conf), len(results_df)]
axes[1].bar(categories, counts, color=['#2ecc71', '#f39c12', '#3498db'], edgecolor='black')
axes[1].set_title('Prediction Confidence Breakdown', fontsize=11)
axes[1].set_ylabel('Number of Players')

plt.suptitle('Scouting Prediction Confidence Analysis', fontsize=13)
plt.tight_layout()
plt.show()

print(f'High-confidence correct predictions (>75% prob): {len(high_conf_correct)}')
print(f'Borderline predictions (40–60% prob): {len(low_conf)} — recommend human scout review')

**Stakeholder-Ready Summary — For the Scouting Department:**

### What the model does
This model analyses 10 performance metrics from a player's early career data and predicts whether they will sustain an NBA career of 5+ years.

### Model performance in plain language
- Out of every 10 players the model marks as **long-term prospects**, approximately **9 will genuinely last 5+ years** (87% Precision)
- However, the model **misses about 45% of genuine long-term players** — flagging them as unlikely to last (low Recall of 55%)

### When to trust the model
| Model says | Trust level | Action |
|---|---|---|
| Player WILL last 5+ years | **High** (87% correct) | Prioritise for contract / development investment |
| Player will NOT last 5+ years | **Moderate** (55% recall) | Do not rule out — supplement with scout evaluation |
| Borderline probability (40–60%) | **Low** | Mandatory human review before decision |

### Top features scouts should focus on
1. **Total Points** — strongest single predictor of longevity
2. **Rebounds** — consistent rebounding signals durability and work rate
3. **Turnovers** — counterintuitively positive; high turnovers indicate high usage and ball-handling role
4. **Steals & Blocks** — defensive effort metrics that correlate with long careers
5. **Field Goal %** — shooting efficiency over volume

### Recommendation for integration into scouting workflow
Use the model as a **first-pass filter**: players flagged as high-probability long-term prospects should be fast-tracked for deeper evaluation. Players the model rates as unlikely to last should still receive human scouting review — the model's conservative nature means it misses real talent.

## 11. Model Limitations & Next Steps

### Limitations

1. **Independence assumption violated** — NBA stats are structurally correlated (points, rebounds, turnovers all relate to playing time). This makes Naive Bayes overconfident in its probability scores.

2. **Low Recall (55%)** — The model misses nearly half of genuine long-term players. For talent identification, this is a significant gap.

3. **No contextual features** — Player position, team quality, age at entry, and injury history are all relevant but absent from this dataset.

4. **Static snapshot** — The model uses career-average stats; it cannot detect improving or declining trajectories.

### Next Steps

- **Compare with other classifiers** — Logistic Regression, Random Forest, and Gradient Boosting would likely outperform Naive Bayes on correlated sports data
- **Threshold tuning** — Lowering the classification threshold from 0.5 to 0.35 would improve Recall at the cost of some Precision, which may be preferable if finding all talent is the priority
- **Add positional data** — Encoding player position could significantly improve class separation
- **Cross-validation** — Use k-fold CV to ensure performance is stable across different data splits

---
**End of Analysis**

*Tools used: Python 3, pandas, NumPy, scikit-learn, Matplotlib, Seaborn*